# Part 11: GNN Virtual Screening of COCONUT Database

Uses trained GNN models (Part 9 GCN + Part 10 GIN) to screen ~154k natural products
from the COCONUT database for MDM2 inhibitory activity.

**Pipeline:** COCONUT SMILES -> graph featurization -> GNN predict -> filter hits

In [ ]:
!git clone https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git
%cd Natural_MDM2_Inhibitor_Discovery_using_ML

In [ ]:
!pip install torch-geometric rdkit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from rdkit import Chem
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Define Model Architectures (same as Part 9 & 10)

In [ ]:
class GCN(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=128, num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)


class GINEncoder(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=300, num_layers=5, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.gin_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.gin_layers.append(nn.Linear(num_node_features, hidden_dim))
        self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.gin_layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        self.eps = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(num_layers)])

    def forward(self, x, edge_index, batch):
        for i in range(self.num_layers):
            neighbor_sum = self._aggregate(x, edge_index)
            x = self.gin_layers[i]((1 + self.eps[i]) * x + neighbor_sum)
            x = self.bn_layers[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def _aggregate(self, x, edge_index):
        src, dst = edge_index
        messages = x[src]
        agg = torch.zeros_like(x)
        agg.scatter_add_(0, dst.unsqueeze(1).expand_as(messages), messages)
        return agg


class MDM2Classifier(nn.Module):
    def __init__(self, encoder, hidden_dim=600, num_classes=2):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x, edge_index, batch):
        return self.classifier(self.encoder(x, edge_index, batch))

## 2. Load Trained Models

In [ ]:
gcn_model = GCN(num_node_features=78, hidden_dim=128, num_classes=2, dropout=0.2).to(device)
gcn_model.load_state_dict(torch.load('gcn_scratch_model.pth', map_location=device))
gcn_model.eval()
print("GCN (Part 9) loaded.")

gin_encoder = GINEncoder(num_node_features=78, hidden_dim=300, num_layers=5, dropout=0.2)
gin_model = MDM2Classifier(gin_encoder, hidden_dim=600, num_classes=2).to(device)
gin_model.load_state_dict(torch.load('pretrained_gin_mdm2.pth', map_location=device))
gin_model.eval()
print("GIN (Part 10) loaded.")

## 3. Load COCONUT Data

In [ ]:
coconut = pd.read_csv('Part_8/screening_results.csv')
print(f"COCONUT compounds: {len(coconut)}")
coconut.head()

## 4. Featurize COCONUT SMILES -> Graphs

In [ ]:
ATOM_CHOICES = {
    'atomic_num': list(range(1, 101)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2, 3],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
}

def one_hot(val, choices):
    enc = [0] * len(choices)
    if val in choices:
        enc[choices.index(val)] = 1
    return enc

def atom_features(atom):
    f = []
    f += one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f += one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f += one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f += one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f += one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic()))
    f.append(int(atom.IsInRing()))
    return (f + [0] * 78)[:78]

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_features = [atom_features(a) for a in mol.GetAtoms()]
    edge_index = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
    if not edge_index:
        edge_index = [[0, 0]]
    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    )

In [ ]:
graphs = []
valid_idx = []
failed = 0

for idx, row in tqdm(coconut.iterrows(), total=len(coconut), desc="Featurizing"):
    g = mol_to_graph(row['canonical_smiles'])
    if g is not None:
        graphs.append(g)
        valid_idx.append(idx)
    else:
        failed += 1

print(f"Converted: {len(graphs)}/{len(coconut)} | Failed: {failed}")

In [ ]:
screen_loader = DataLoader(graphs, batch_size=256, shuffle=False)
print(f"Batches: {len(screen_loader)}")

## 5. Screen with GCN (Part 9)

In [ ]:
gcn_preds, gcn_probs = [], []

gcn_model.eval()
with torch.no_grad():
    for batch in tqdm(screen_loader, desc="GCN Screening"):
        batch = batch.to(device)
        out = gcn_model(batch.x, batch.edge_index, batch.batch)
        probs = F.softmax(out, dim=1)
        gcn_probs.extend(probs[:, 1].cpu().numpy())
        gcn_preds.extend(out.argmax(dim=1).cpu().numpy())

gcn_preds = np.array(gcn_preds)
gcn_probs = np.array(gcn_probs)
print(f"GCN screening done. Predicted active: {(gcn_preds == 1).sum()}")

## 6. Screen with GIN (Part 10)

In [ ]:
gin_preds, gin_probs = [], []

gin_model.eval()
with torch.no_grad():
    for batch in tqdm(screen_loader, desc="GIN Screening"):
        batch = batch.to(device)
        out = gin_model(batch.x, batch.edge_index, batch.batch)
        probs = F.softmax(out, dim=1)
        gin_probs.extend(probs[:, 1].cpu().numpy())
        gin_preds.extend(out.argmax(dim=1).cpu().numpy())

gin_preds = np.array(gin_preds)
gin_probs = np.array(gin_probs)
print(f"GIN screening done. Predicted active: {(gin_preds == 1).sum()}")

## 7. Assemble Results

In [ ]:
# Build results DataFrame for valid molecules only
results = coconut.iloc[valid_idx][['identifier', 'canonical_smiles']].copy()

# Add RF predictions from Part 8
rf_orig = pd.read_csv('Part_8/screening_results.csv')
results['rf_prediction'] = rf_orig.iloc[valid_idx]['prediction'].values
results['rf_prob_active'] = rf_orig.iloc[valid_idx]['prob_class_1'].values

# Add GNN predictions
results['gcn_prediction'] = gcn_preds
results['gcn_prob_active'] = gcn_probs
results['gin_prediction'] = gin_preds
results['gin_prob_active'] = gin_probs

# Consensus: active if majority of 3 models predict active
results['consensus'] = ((results['rf_prediction'] + results['gcn_prediction'] + results['gin_prediction']) >= 2).astype(int)

print(f"Total screened: {len(results)}")
print(f"\nPredicted active by model:")
print(f"  RF:   {results['rf_prediction'].sum()}")
print(f"  GCN:  {results['gcn_prediction'].sum()}")
print(f"  GIN:  {results['gin_prediction'].sum()}")
print(f"  Consensus (2/3): {results['consensus'].sum()}")
results.head(10)

In [ ]:
results.to_csv('gnn_screening_results.csv', index=False)

In [ ]:
print('Saved: gnn_screening_results.csv')

## 8. Filter High-Confidence Hits

In [ ]:
# Consensus hits (2/3 models agree)
consensus_hits = results[results['consensus'] == 1].copy()
consensus_hits = consensus_hits.sort_values('gin_prob_active', ascending=False)
print(f"Consensus hits: {len(consensus_hits)}")
consensus_hits

In [ ]:
consensus_hits.to_csv('gnn_consensus_hits.csv', index=False)

In [ ]:
# GIN-only high confidence (prob > 0.6)
gin_high_conf = results[(results['gin_prediction'] == 1) & (results['gin_prob_active'] > 0.6)].copy()
gin_high_conf = gin_high_conf.sort_values('gin_prob_active', ascending=False)
print(f"GIN high-confidence hits (>0.6): {len(gin_high_conf)}")
gin_high_conf

## 9. Compare GNN vs RF Screening

In [ ]:
# How many compounds do all 3 models agree on?
all_agree = results[(results['rf_prediction'] == 1) & (results['gcn_prediction'] == 1) & (results['gin_prediction'] == 1)]
print(f"All 3 models agree (active): {len(all_agree)}")

# Only GNN finds (not RF)
gnn_only = results[(results['gcn_prediction'] == 1) | (results['gin_prediction'] == 1) & (results['rf_prediction'] == 0)]
gnn_only_unique = gnn_only[~gnn_only.index.isin(results[results['rf_prediction'] == 1].index)]
print(f"GNN-only finds (not RF): {len(gnn_only_unique)}")

# Only RF finds (not GNN)
rf_only = results[(results['rf_prediction'] == 1) & (results['gcn_prediction'] == 0) & (results['gin_prediction'] == 0)]
print(f"RF-only finds (not GNN): {len(rf_only)}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
categories = ['RF only', 'GCN only', 'GIN only', 'RF+GCN', 'RF+GIN', 'GCN+GIN', 'All 3']
counts = [
    len(rf_only),
    len(results[(results['gcn_prediction']==1) & (results['rf_prediction']==0) & (results['gin_prediction']==0)]),
    len(results[(results['gin_prediction']==1) & (results['rf_prediction']==0) & (results['gcn_prediction']==0)]),
    len(results[(results['rf_prediction']==1) & (results['gcn_prediction']==1) & (results['gin_prediction']==0)]),
    len(results[(results['rf_prediction']==1) & (results['gin_prediction']==1) & (results['gcn_prediction']==0)]),
    len(results[(results['gcn_prediction']==1) & (results['gin_prediction']==1) & (results['rf_prediction']==0)]),
    len(all_agree)
]
colors = ['steelblue', 'coral', 'green', 'mediumpurple', 'orange', 'teal', 'gold']
ax.bar(categories, counts, color=colors, edgecolor='black')
ax.set_ylabel('Number of Compounds')
ax.set_title('Model Agreement on COCONUT Screening')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('gnn_vs_rf_screening.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save SMILES for Downstream Docking

In [ ]:
# Save consensus hit SMILES
with open('gnn_consensus_smiles.txt', 'w') as f:
    for smi in consensus_hits['canonical_smiles']:
        f.write(smi + '\n')
print(f"Saved {len(consensus_hits)} SMILES to gnn_consensus_smiles.txt")

In [ ]:
print("\n=== Part 11 Complete ===")
print("Outputs:")
print("  - gnn_screening_results.csv (all compounds with RF/GCN/GIN predictions)")
print("  - gnn_consensus_hits.csv (2/3 models agree = active)")
print("  - gnn_consensus_smiles.txt (SMILES for docking)")
print("  - gnn_vs_rf_screening.png (comparison plot)")